# Windows MT5 Market Data Export

Run this notebook on the Windows laptop where MetaTrader 5 is installed, open, and logged in to the broker account shown in Market Watch.

It exports OHLC candles for the exact 28 instruments in the supplied screenshot. Broker suffixes such as `.pc` and `.sc` are preserved. Output is compressed CSV plus a JSON manifest under `data/raw/mt5_export`.

> MT5 returns only history available to the terminal. In MT5, increase **Tools → Options → Charts → Max bars in chart** and allow the terminal to download history if an old start date returns fewer rows than expected.

## 1. Editable settings

Change `START_DATE` or `TIMEFRAMES` before running all cells. If multiple MT5 terminals are installed, set `MT5_TERMINAL_PATH` to the correct `terminal64.exe`; otherwise leave it as `None`.

In [6]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "mt5_export"

START_DATE = "2026-01-01"  # UTC, YYYY-MM-DD
TIMEFRAMES = ["M1", "M5", "M15", "H1", "H4", "D1"]
MT5_TERMINAL_PATH = None  # Example: r"C:\\Program Files\\Broker MT5\\terminal64.exe"

SYMBOLS = [
    "EURUSD", "GBPUSD", "USDJPY", "AUDUSD", "NZDUSD", "USDCAD",
    "XAUUSD.pc", "NAS100", "BTCUSD.sc", "USDCHF.pc", "GBPJPY.pc",
    "EURJPY.pc", "SP500", "AUDCAD.pc", "AUDCHF.pc", "AUDJPY.pc",
    "CADCHF.pc", "CADJPY.pc", "CHFJPY.pc", "COPPER-C", "EURAUD.pc",
    "EURCAD.pc", "EURCHF.pc", "EURGBP.pc", "GBPAUD.pc", "GBPCAD.pc",
    "GBPCHF.pc", "USOUSD.pc",
]

assert len(SYMBOLS) == 28 and len(set(SYMBOLS)) == 28
print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
print(f"System:  {platform.platform()}")
print(f"Export:  {len(SYMBOLS)} symbols × {len(TIMEFRAMES)} timeframes")

Project: D:\viet\quant_trading\sb_trading_system
Python:  D:\viet\quant_trading\sb_trading_system\.venv\Scripts\python.exe
System:  Windows-10-10.0.19044-SP0
Export:  28 symbols × 6 timeframes


## 2. Verify the MT5 connection and symbols

Install dependencies in the selected notebook kernel if needed:

```python
%pip install -r ../requirements-win-mt5.txt
```

Restart the kernel after installing `MetaTrader5`.

In [7]:
try:
    import MetaTrader5 as mt5
except ImportError as exc:
    raise ImportError(
        "MetaTrader5 is not installed in this notebook kernel. Run "
        f"{sys.executable} -m pip install -r {PROJECT_ROOT / 'requirements-win-mt5.txt'}"
    ) from exc

init_kwargs = {"path": MT5_TERMINAL_PATH} if MT5_TERMINAL_PATH else {}
if not mt5.initialize(**init_kwargs):
    raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}")

try:
    account = mt5.account_info()
    terminal = mt5.terminal_info()
    if account is None:
        raise RuntimeError("MT5 is connected to a terminal but no trading account is logged in.")

    checks = []
    for symbol in SYMBOLS:
        info = mt5.symbol_info(symbol)
        selected = bool(info) and mt5.symbol_select(symbol, True)
        tick = mt5.symbol_info_tick(symbol) if selected else None
        checks.append({
            "symbol": symbol,
            "available": info is not None,
            "selected": selected,
            "digits": info.digits if info else None,
            "last_tick_utc": pd.to_datetime(tick.time, unit="s", utc=True) if tick else None,
        })

    symbol_check = pd.DataFrame(checks)
    print(f"Account: {account.login} | Server: {account.server}")
    print(f"Terminal: {terminal.path if terminal else 'unknown'}")
    display(symbol_check)

    missing = symbol_check.loc[~symbol_check["selected"], "symbol"].tolist()
    if missing:
        raise RuntimeError(
            "Unavailable MT5 symbols: " + ", ".join(missing) +
            ". Compare them with Market Watch and update SYMBOLS exactly, including suffixes."
        )
finally:
    mt5.shutdown()

Account: 26448239 | Server: VantageMarkets-Live 21
Terminal: C:\Program Files\MetaTrader 5_2


,symbol,available,selected,digits,last_tick_utc
0,EURUSD,True,True,5,2026-07-24 23:56:52+00:00
1,GBPUSD,True,True,5,2026-07-24 23:56:56+00:00
2,USDJPY,True,True,3,2026-07-24 23:56:54+00:00
3,AUDUSD,True,True,5,2026-07-24 23:56:59+00:00
4,NZDUSD,True,True,5,2026-07-24 23:56:55+00:00
5,USDCAD,True,True,5,2026-07-24 23:56:49+00:00
6,XAUUSD.pc,True,True,2,2026-07-24 23:56:59+00:00
7,NAS100,True,True,2,2026-07-24 23:59:59+00:00
8,BTCUSD.sc,True,True,2,2026-07-26 20:01:48+00:00
9,USDCHF.pc,True,True,5,2026-07-24 23:56:45+00:00


## 3. Export historical candles

The exporter performs the same symbol preflight again, downloads each symbol/timeframe with `copy_rates_range`, normalizes timestamps to UTC, and writes one `.csv.gz` file per dataset.

In [8]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "export_mt5_candles.py"),
    "--output-dir", str(OUTPUT_DIR),
    "--symbols", ",".join(SYMBOLS),
    "--timeframes", ",".join(TIMEFRAMES),
    "--start", START_DATE,
]
if MT5_TERMINAL_PATH:
    cmd.extend(["--terminal-path", MT5_TERMINAL_PATH])

print("Starting MT5 export...")
result = subprocess.run(cmd, cwd=PROJECT_ROOT)
if result.returncode != 0:
    raise RuntimeError(f"MT5 export failed with exit code {result.returncode}")
print(f"Export complete: {OUTPUT_DIR}")

Starting MT5 export...
Export complete: D:\viet\quant_trading\sb_trading_system\data\raw\mt5_export


## 4. Validate the export

In [9]:
manifest_path = OUTPUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
summary = pd.DataFrame(manifest["files"])

expected_files = len(SYMBOLS) * len(TIMEFRAMES)
if len(summary) != expected_files:
    raise RuntimeError(f"Expected {expected_files} exports, found {len(summary)}")
if (summary["rows"] == 0).any():
    empty = summary.loc[summary["rows"] == 0, ["symbol", "timeframe"]]
    raise RuntimeError(f"Empty datasets found:\n{empty.to_string(index=False)}")

print(f"Range requested: {manifest['date_from']} → {manifest['date_to']}")
print(f"Files: {len(summary)} | Total candles: {manifest['total_rows']:,}")
display(summary.pivot(index="symbol", columns="timeframe", values="rows").fillna(0).astype(int))

Range requested: 2026-01-01T00:00:00+00:00 → 2026-07-26T17:01:51.532777+00:00
Files: 168 | Total candles: 7,615,735


timeframe,D1,H1,H4,M1,M15,M5
symbol,,,,,,
AUDCAD.pc,146,3504,876,209908,14016,42047
AUDCHF.pc,146,3504,876,209917,14016,42048
AUDJPY.pc,146,3504,876,209909,14016,42047
AUDUSD,146,3504,876,209918,14016,42048
BTCUSD.sc,207,4935,1237,295959,19736,59201
CADCHF.pc,146,3504,876,209917,14016,42048
CADJPY.pc,146,3504,876,209909,14016,42048
CHFJPY.pc,146,3504,876,209912,14016,42048
COPPER-C,145,3321,868,198881,13268,39791


## 5. Preview exported data

Columns are `broker_symbol`, `timeframe`, UTC `candle_time`, OHLC, `tick_volume`, broker spread, and `real_volume`. In spot FX, MT5 volume is commonly tick volume rather than centralized traded volume.

In [10]:
sample_row = summary.query("symbol == 'EURUSD' and timeframe == 'M5'").iloc[0]
sample = pd.read_csv(sample_row["path"], parse_dates=["candle_time"])
print(sample_row["path"])
print(f"Rows: {len(sample):,} | {sample['candle_time'].min()} → {sample['candle_time'].max()}")
display(sample.tail(20))

D:\viet\quant_trading\sb_trading_system\data\raw\mt5_export\EURUSD_M5_2026-01-01_to_now.csv.gz
Rows: 42,048 | 2026-01-02 00:00:00+00:00 → 2026-07-24 23:55:00+00:00


,broker_symbol,timeframe,candle_time,open,high,low,close,tick_volume,spread,real_volume
42028,EURUSD,M5,2026-07-24 22:20:00+00:00,1.13696,1.13702,1.13681,1.13685,220,13,0
42029,EURUSD,M5,2026-07-24 22:25:00+00:00,1.13684,1.13686,1.13675,1.13681,194,13,0
42030,EURUSD,M5,2026-07-24 22:30:00+00:00,1.13680,1.13696,1.13675,1.13686,262,13,0
42031,EURUSD,M5,2026-07-24 22:35:00+00:00,1.13686,1.13717,1.13684,1.13711,352,13,0
42032,EURUSD,M5,2026-07-24 22:40:00+00:00,1.13711,1.13717,1.13706,1.13708,247,13,0
42033,EURUSD,M5,2026-07-24 22:45:00+00:00,1.13708,1.13711,1.13697,1.13704,174,13,0
42034,EURUSD,M5,2026-07-24 22:50:00+00:00,1.13704,1.13707,1.13697,1.13701,159,13,0
42035,EURUSD,M5,2026-07-24 22:55:00+00:00,1.13702,1.13704,1.13693,1.13693,220,13,0
42036,EURUSD,M5,2026-07-24 23:00:00+00:00,1.13694,1.13700,1.13694,1.13698,167,13,0
42037,EURUSD,M5,2026-07-24 23:05:00+00:00,1.13698,1.13699,1.13692,1.13695,130,13,0


## Output

Keep or copy the entire folder:

```text
data/raw/mt5_export
```

The compressed CSV files can be imported into the platform later with:

```bat
python scripts\import_candles_to_files.py data\raw\mt5_export
```